# North Dakota — Title 26.1 (Insurance) → `data/north_dakota/ins_codes/*.md`

North Dakota’s **insurance** provisions are **Title 26.1 — Insurance** of the **North Dakota Century Code (N.D.C.C.)**. On **Justia**, the browse root is **[`/codes/north-dakota/title-26-1/`](https://law.justia.com/codes/north-dakota/title-26-1/)** (URL uses **`26-1`**; headings read **Title 26.1**).

**Unlike many other states on Justia**, Title 26.1 is published **by chapter**, not per **`/section-…`** URL: each **chapter** page (e.g. **`…/chapter-26-1-01/`**) contains the **entire chapter** (all sections in one HTML page).

**Cloudflare** often blocks plain **`httpx`**; this notebook uses **`curl_cffi`** with **`impersonate="chrome120"`**.

**Discovery:** Parse **`div.primary-content`** on the title index; collect **`/codes/north-dakota/title-26-1/chapter-…`** links (yearless only; ~**118** chapters).

**Download:** one **.md** file per **chapter**. Files: **`ND_cc_ch_<slug>.md`** with underscores (e.g. **`26-1-01`** → **`ND_cc_ch_26_1_01.md`**).

**Config:** **`MAX_CHAPTERS`** (**0** = all), **`REUSE_DISCOVERED_URLS`**, **`_nd_title26_1_chapter_urls.txt`**.

Run with **`ins_ipynb/`** as cwd, then **`python -m app.ingest`** from the project root.


In [1]:
%pip install -q curl_cffi beautifulsoup4


You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

BASE = "https://law.justia.com"
PATH_PREFIX = "/codes/north-dakota/title-26-1"
TITLE_INDEX = f"{BASE}{PATH_PREFIX}/"

OUT_DIR = Path("data") / "north_dakota" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CURL_IMPERSONATE = "chrome120"
REQUEST_DELAY_SEC = 0.12
TIMEOUT = 60.0

MAX_CHAPTERS = 0

SKIP_EXISTING = True

DISCOVERED_LIST = OUT_DIR / "_nd_title26_1_chapter_urls.txt"
REUSE_DISCOVERED_URLS = True


In [3]:
def curl_get(url: str) -> str:
    time.sleep(REQUEST_DELAY_SEC)
    r = curl_requests.get(url, impersonate=CURL_IMPERSONATE, timeout=TIMEOUT)
    r.raise_for_status()
    return r.text


def path_key(u: str) -> str:
    return urlparse(u).path.rstrip("/")


def norm_chapter_url(url: str) -> str:
    u = url.split("?")[0].rstrip("/")
    return u + "/"


def discover_chapter_urls() -> list[str]:
    """Chapter index links on the Title 26.1 page (yearless paths only)."""
    html = curl_get(TITLE_INDEX)
    soup = BeautifulSoup(html, "html.parser")
    pc = soup.select_one("div.primary-content") or soup
    chapters: set[str] = set()
    for a in pc.find_all("a", href=True):
        absu = norm_chapter_url(urljoin(TITLE_INDEX, a["href"]))
        p = path_key(absu).lower()
        if not p.startswith(PATH_PREFIX + "/chapter-"):
            continue
        tail = absu.split("/codes/north-dakota/", 1)[-1]
        if tail.startswith("20"):
            continue
        if "/section-" in p:
            continue
        chapters.add(absu)
    return sorted(chapters, key=chapter_label_sort_key)


def chapter_slug_from_url(url: str) -> str:
    leaf = path_key(url).rsplit("/", 1)[-1]
    low = leaf.lower()
    if not low.startswith("chapter-"):
        raise ValueError(f"not a chapter URL: {url!r}")
    return leaf[len("chapter-") :]


def chapter_label_sort_key(url: str) -> tuple:
    slug = chapter_slug_from_url(url)
    out: list[tuple[int, int | str]] = []
    for part in slug.split("-"):
        if part.isdigit():
            out.append((0, int(part)))
        else:
            out.append((1, part.lower()))
    return tuple(out)


def slug_to_filename(slug: str) -> str:
    safe = slug.replace("-", "_")
    return f"ND_cc_ch_{safe}.md"


def chapter_display(slug: str) -> str:
    if slug.startswith("26-1-"):
        return "26.1-" + slug[5:]
    return slug


def extract_primary_text(html: str) -> tuple[str, str]:
    soup = BeautifulSoup(html, "html.parser")
    title_el = soup.find("title")
    title_txt = title_el.get_text(strip=True) if title_el else ""
    pc = soup.select_one("div.primary-content")
    if pc:
        text = pc.get_text("\n", strip=True)
    else:
        main = soup.find("main") or soup.find("article")
        text = main.get_text("\n", strip=True) if main else soup.get_text("\n", strip=True)
    return title_txt, text


def strip_justia_boilerplate(text: str) -> str:
    drop_prefixes = (
        "Go to Previous Versions",
        "View All Versions",
        "Learn more",
        "This media-neutral citation",
    )
    lines = text.split("\n")
    out: list[str] = []
    skip_until_substantive = True
    for line in lines:
        s = line.strip()
        if not s:
            if not skip_until_substantive:
                out.append("")
            continue
        if any(s.startswith(p) for p in drop_prefixes):
            continue
        if s.startswith("20") and ("N.D. Cent" in s or "North Dakota Cent" in s):
            continue
        if s in {"Next", "Previous", "Universal Citation:", "Download as PDF"}:
            continue
        skip_until_substantive = False
        out.append(s)
    return "\n".join(out).strip()


def download_title_26_1() -> dict[str, int]:
    if REUSE_DISCOVERED_URLS and DISCOVERED_LIST.exists() and DISCOVERED_LIST.stat().st_size > 50:
        raw = [ln.strip() for ln in DISCOVERED_LIST.read_text(encoding="utf-8").splitlines() if ln.strip()]
        all_urls = sorted(raw, key=chapter_label_sort_key)
        print(f"Loaded {len(all_urls)} chapter URLs from {DISCOVERED_LIST.name} (skipped discovery)")
    else:
        found = discover_chapter_urls()
        print(f"Discovered {len(found)} chapter URLs under Title 26.1")
        all_urls = sorted(found, key=chapter_label_sort_key)
        DISCOVERED_LIST.write_text("\n".join(all_urls) + "\n", encoding="utf-8")

    todo = all_urls if not MAX_CHAPTERS else all_urls[:MAX_CHAPTERS]
    if MAX_CHAPTERS:
        print(f"Limited downloads to first {len(todo)} chapters (MAX_CHAPTERS)")

    wrote = skipped = failed = 0
    for i, ch_url in enumerate(todo, 1):
        slug = chapter_slug_from_url(ch_url)
        dest = OUT_DIR / slug_to_filename(slug)
        if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 200:
            skipped += 1
        else:
            try:
                html = curl_get(ch_url)
                head_t, body_t = extract_primary_text(html)
                body_t = strip_justia_boilerplate(body_t)
                disp = chapter_display(slug)
                title = head_t or f"North Dakota Century Code — Chapter {disp}"
                md = (
                    f"# {title}\n\n"
                    f"**North Dakota Century Code — Title 26.1 (Insurance)**\n\n"
                    f"**Source (Justia mirror):** {ch_url}\n\n"
                    f"**Verify on official site:** [ND Legislative Branch — Century Code](https://www.legis.nd.gov/general-information/north-dakota-century-code-online)\n\n"
                    f"**Chapter (URL slug):** {slug}\n\n"
                    f"**Chapter (display):** {disp}\n\n"
                    f"---\n\n"
                    f"{body_t}\n"
                )
                dest.write_text(md, encoding="utf-8")
                wrote += 1
            except Exception as e:
                print(f"FAIL {slug}: {e}")
                failed += 1
        if i % 25 == 0:
            print(f"… {i}/{len(todo)} (wrote={wrote} skipped={skipped} failed={failed})")

    print(f"Done. wrote={wrote} skipped={skipped} failed={failed} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped, "failed": failed}


download_title_26_1()


Discovered 118 chapter URLs under Title 26.1
… 25/118 (wrote=25 skipped=0 failed=0)
… 50/118 (wrote=50 skipped=0 failed=0)
… 75/118 (wrote=75 skipped=0 failed=0)
… 100/118 (wrote=100 skipped=0 failed=0)
Done. wrote=118 skipped=0 failed=0 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/north_dakota/ins_codes


{'wrote': 118, 'skipped': 0, 'failed': 0}

## Next step

`python -m app.ingest` from the project root.
